## 第11章 文件

### 1.打开文件

- **打开文件**：`open(file, mode='r', encoding='utf-8', errors='ignore', newline=None)`。
    - file：文件名，必填。若文件与脚本同目录可直接传文件名，否则需传入完整路径。
    - mode：打开模式，详细见下表。
    - encoding：编码，默认`'utf-8'`。
    - errors：Unicode错误处理策略，默认`'ignore'`，忽略错误。
    - newline：换行符。读取时自动将`\r`、`\r\n`转为`\n`，写入时自动将`\n`转为系统默认行尾符。

- **打开模式**：详见参数含义如下：

| 值    | 描述         | 说明                                       |
| ----- | ------------ | ------------------------------------------ |
| `'r'` | 读取模式     | 默认值，文件不存在则报错                   |
| `'w'` | 写入模式     | 文件不存在则创建；**存在则截断（清空）** ⚠️ |
| `'x'` | 独占写入模式 | 文件已存在则引发 `FileExistsError`         |
| `'a'` | 附加模式     | 在文件末尾追加，不存在则创建               |
| `'b'` | 二进制模式   | 与其他模式结合使用（如 `'rb'`、`'wb'`）    |
| `'t'` | 文本模式     | 默认值，与其他模式结合使用                 |
| `'+'` | 读写模式     | 与其他模式结合使用（如 `'r+'`、`'w+'`）    |

⚠️ **`'r+'` vs `'w+'`**：前者不截断文件，后者截断！


### 2.操作文件

- **读取操作**：
  - `read(n)`：读取`n`个字符(二进制模式下为`n`字节)，不指定`n`则读取剩余全部内容。
  - `readline(n)`：读取一行(到换行符为止)，可选参数`n`指定最多读取字符数。
  - `readlines()`：读取所有行，返回包含每行内容的字符串列表，每行保留换行符。
- **写入操作**：
  - `write(s)`：写入字符串`s`，返回写入的字符数。
  - `writelines(lines)`：写入字符串列表`lines`，返回写入的字符数。**不会自动添加换行符，需自行在字符串中加`\n`**。
  - `flush()`：刷新缓冲缓冲区，将所有写入的数据写入文件。
- **随机存取**：
  - `seek(offset[, whence])`：将文件指针从当前位置移动到`offset`和`whence`指定的位置。
    - 参数`whence`默认为io.SEEK_SET​，相对于文件开头移动，指定为io.SEEK_CUR时相对于当前位置移动，指定为io.SEEK_END时相对于文件末尾移动。
  - `tell()`：返回当前文件指针位置。
- **关闭文件**：
  - `close()`：关闭文件。程序退出时可能自动关闭文件，但显式关闭可避免文件锁定；写入文件后**必须关闭**，否则缓冲数据可能因程序崩溃丢失。
  - 安全关闭方式：使用`with open("file") as f:`语句打开文件，语句块结束后自动关闭文件，即便发生异常也生效。

In [ ]:
# read(n)
f = open('./res/test1.txt')         # 打开文件，只读模式
print(f.read(7))                    # 读取7个字符, 输出：Welcome
print(f.read(4))                    # 继续读取4个字符, 输出：to
f.close()                           # 关闭文件

# read()
f = open('./res/test1.txt')
print(f.read())                     # 读取所有字符
f.close()

# readline()
f = open('./res/test1.txt')
for i in range(3):
    print(f'{i}: {f.readline()}', end='')   # 逐行读取文件内容(包含换行符)
f.close()
print()

# readlines()
import pprint
f = open('./res/test1.txt')
pprint.pprint(f.readlines())         # 读取所有行(包含换行符)，返回列表
f.close()

# write()
f = open('./res/test2.txt', 'w')     # 打开文件，写入模式
f.write('this\nis no\nhaiku')        # 写入内容
f.close()

# writelines()
f = open('./res/test2.txt')          # 打开文件，只读模式
lines = f.readlines()
f.close()
lines[1] = "isn't a\n"
f = open('./res/test2.txt', 'w')     # 打开文件，写入模式
f.writelines(lines)                  # 写入字符串列表
f.close()

# seek()
f = open('./res/test3.txt', 'w')
f.write('01234567890123456789')
f.seek(5)                            # 从文件开头移动到位置5
f.write('Hello, World!')             # 从位置5开始覆写文件内容，文件内容变为：01234Hello, World!89
f.close()

### 3.迭代文件

- 每次一个字符：read + while
- 每次一行：readline + while
- 读取所有内容：read / readlines + for
- 延迟行迭代：使用 fileinput 模块
- 文件迭代器：for line in f

**五种方法对比**

| 方法                 | 内存占用           | 代码简洁度 | 推荐程度 |
| -------------------- | ------------------ | ---------- | -------- |
| `read(1)` + while    | 低                 | ❌ 啰嗦     | ⭐        |
| `readline()` + while | 低                 | ❌ 啰嗦     | ⭐        |
| `read()` + for       | **高**（全部读入） | 中         | ⭐⭐       |
| `readlines()` + for  | **高**（全部读入） | 中         | ⭐⭐       |
| `fileinput.input()`  | 低                 | 中         | ⭐⭐⭐      |
| **`for line in f`**  | **低**（惰性）     | ✅ 最简     | ⭐⭐⭐⭐⭐    |

In [ ]:
print('1.每次一个字符')
with open('./res/test1.txt', 'r') as f:
    while True:
        char = f.read(1)
        if not char: break
        print(char, end='')

print('\n2.每次一行')
with open('./res/test1.txt', 'r') as f:
    while True:
        line = f.readline()
        if not line: break
        print(line, end='')


print('\n3.读取所有内容(read)')
with open('./res/test1.txt', 'r') as f:
    for char in f.read():
        print(char, end='')

print('\n4.读取所有内容(readlines)')
with open('./res/test1.txt', 'r') as f:
    for line in f.readlines():
        print(line, end='')

print('\n5.使用fileinput模块')
import fileinput
for line in fileinput.input('./res/test1.txt'):
    print(line, end='')

print('\n6.文件迭代器')
with open('./res/test1.txt', 'r') as f:
    for line in f:
        print(line, end='')

### 4.本章小结

**核心知识脉络**：

```text
文件操作
│
├── 打开文件
│   ├── open(filename, mode)
│   ├── 模式：r/w/x/a/b/t/+ ⭐
│   ├── 编码：encoding='utf-8'
│   └── 换行：newline=''
│
├── 读取方法
│   ├── f.read(n)     → 读取n个字符
│   ├── f.read()      → 读取全部
│   ├── f.readline()  → 读取一行
│   └── f.readlines() → 所有行的列表
│
├── 写入方法
│   ├── f.write(str)       → 写入字符串
│   ├── f.writelines(list) → 写入字符串列表
│
├── 关闭文件 ⭐
│   ├── f.close()           → 手动关闭
│   ├── f.flush()           → 刷新缓冲区
│   └── with open() as f:   → 自动关闭（推荐）
│
├── 随机存取
│   ├── f.seek(offset)  → 移动位置
│   └── f.tell()        → 返回当前位置
│
└── 迭代文件内容 ⭐
    ├── for line in f          → 最推荐 ⭐⭐⭐⭐⭐
    ├── fileinput.input()      → 多文件处理
    ├── f.readlines() + for    → 小文件
    ├── f.readline() + while   → 大文件逐行
    └── f.read(1) + while      → 逐字符
```

**文件方法速查表**

| 方法                | 作用        | 返回值           |
| ------------------- | ----------- | ---------------- |
| `f.read(n)`         | 读取n个字符 | 字符串           |
| `f.read()`          | 读取全部    | 字符串           |
| `f.readline()`      | 读取一行    | 字符串（含`\n`） |
| `f.readlines()`     | 读取所有行  | 列表             |
| `f.write(s)`        | 写入字符串  | 写入字符数       |
| `f.writelines(lst)` | 写入行列表  | `None`           |
| `f.close()`         | 关闭文件    | `None`           |
| `f.flush()`         | 刷新缓冲区  | `None`           |
| `f.seek(offset)`    | 移动位置    | 新位置           |
| `f.tell()`          | 当前位置    | 整数             |

**最佳实践清单**

1. **始终用 `with` 语句打开文件** —— 自动关闭，防异常丢数据
2. **读取文件首选 `for line in f`** —— 最简洁、最高效
3. **写模式 `'w'` 会清空文件** —— 除非有意为之，否则用 `'a'` 追加
4. **二进制文件用 `'b'` 模式** —— 图片、音频等不做编码转换
5. **`writelines` 不加换行符** —— 需要手动在字符串中加 `\n`